# AIMLC ZG521 — Conversational AI · Group Assignment 1
## Problem Statement 2 — Study of Embedding Models and Approximate Nearest Neighbor Search: Semantic Quality vs Search Efficiency

**Group 129** · Total: 10 Marks · Deadline: 28 Aug 2026

## Student Details

| Name | BITS ID | Email |
|---|---|---|
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |

## Contribution by Each Student

| Member | Area | Task(s) | Section(s) done |
|---|---|---|---|
| P1 | Dataset & embedding + Visualization & reporting | T1, T2, T8, report assembly | *TODO — name* |
| P2 | Similarity matrix comparison + kNN baseline | T3, T4 | *TODO — name* |
| P3 | HNSW vs IVF | T5 | *TODO — name* |
| P4 | Evaluation & analysis | T6, T7 | *TODO — name* |

(Per `ass-1/ROADMAP.md` day-by-day execution plan.)

## Problem Statement

Study of embedding models and approximate nearest neighbor (ANN) search, comparing **semantic quality vs search efficiency**:
- **Module 1** — Dataset and embedding preparation (1 mark)
- **Module 2** — Similarity metrics and exact retrieval (3 marks)
- **Module 3** — ANN search experiment: HNSW vs IVF (3 marks)
- **Module 4** — Embedding quality analysis and final recommendation (3 marks)

Full task-by-task plan: `ass-1/ROADMAP.md`.

## Tools and Libraries Used

- **Python 3.10**
- `datasets` (Hugging Face) — loading the BEIR-format retrieval dataset
- `pandas`, `numpy` — data handling
- `sentence-transformers` — encoder embedding models (Task 2 onward)
- `faiss-cpu` — HNSW / IVF ANN indexes (Task 5 onward)
- `matplotlib` — plots (Task 6 onward)
- Environment: course-provided remote system (see `ass-1/ROADMAP.md` — "Resolved clarifications")

In [1]:
import sys, time, json, random
import numpy as np
import pandas as pd

random.seed(129)   # Group 129 — fixed seed for reproducibility
np.random.seed(129)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

Python: 3.10.12
pandas: 2.3.3
numpy : 2.2.6


---
# Task 1 — Corpus and Query Dataset Preparation (0.5 Marks)

**Dataset chosen: `SciFact`** (from the BEIR benchmark — Thakur et al., 2021), via its scientific-claim-verification source (Wadden et al., 2020).

**Why SciFact:** it is a publicly available, BEIR-format dataset that ships a document corpus, a query set, and **query-document relevance labels (qrels)** together, in exactly the shape this task asks for — no relevance judgments need to be hand-built. Its corpus (~5,183 passages) and query set (~300 claims) both comfortably clear the assignment's 1,000-passage / 50-query minimums.

In [2]:
import os
from datasets import load_dataset

DATA_DIR = "data"

def load_beir_scifact():
    """
    Load the SciFact BEIR dataset (corpus, queries, qrels): local cache under
    data/ if already present, otherwise a live download from the Hugging Face
    Hub. Queries are then filtered to only those with a relevance judgment, so
    "relevance information for each query" is literally true for the result.
    """
    cache_files = [f"{DATA_DIR}/corpus.jsonl", f"{DATA_DIR}/queries.jsonl", f"{DATA_DIR}/qrels.tsv"]
    if all(os.path.exists(p) for p in cache_files):
        corpus_df = pd.read_json(cache_files[0], lines=True)
        queries_df = pd.read_json(cache_files[1], lines=True)
        qrels_df = pd.read_csv(cache_files[2], sep="\t")
        source = "local-cache"
    else:
        corpus_ds = load_dataset("BeIR/scifact", "corpus", split="corpus")
        queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
        qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

        corpus_df = corpus_ds.to_pandas().rename(columns={"_id": "doc_id"})
        queries_df = queries_ds.to_pandas().rename(columns={"_id": "query_id"})
        qrels_df = qrels_ds.to_pandas()
        qrels_df.columns = ["query_id", "doc_id", "relevance"]
        source = "live-huggingface"

    corpus_df["doc_id"] = corpus_df["doc_id"].astype(str)
    queries_df["query_id"] = queries_df["query_id"].astype(str)
    qrels_df["query_id"] = qrels_df["query_id"].astype(str)
    qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)

    # Keep only queries with >=1 relevance judgment (standard BEIR evaluation practice)
    labelled_ids = set(qrels_df["query_id"].unique())
    before = len(queries_df)
    queries_df = queries_df[queries_df["query_id"].isin(labelled_ids)].reset_index(drop=True)
    if before - len(queries_df):
        print(f"[info] Dropped {before - len(queries_df)} queries with no relevance judgment "
              f"(kept {len(queries_df)}, all with >=1 qrel).")

    return corpus_df, queries_df, qrels_df, source


corpus_df, queries_df, qrels_df, data_source = load_beir_scifact()
print(f"\ndata_source = {data_source!r}")
print(f"corpus  : {len(corpus_df)} passages")
print(f"queries : {len(queries_df)} queries (all with >=1 relevance judgment)")
print(f"qrels   : {len(qrels_df)} relevance judgments")

/sessions/inspiring-vibrant-rubin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



data_source = 'local-cache'
corpus  : 5183 passages
queries : 300 queries (all with >=1 relevance judgment)
qrels   : 339 relevance judgments


In [3]:
# Validate against the assignment's stated minimums
MIN_CORPUS, MIN_QUERIES = 1000, 50

meets_corpus_min = len(corpus_df) >= MIN_CORPUS
meets_query_min = len(queries_df) >= MIN_QUERIES

print(f"Corpus  >= {MIN_CORPUS}: {meets_corpus_min}  ({len(corpus_df)} passages)")
print(f"Queries >= {MIN_QUERIES}: {meets_query_min}  ({len(queries_df)} queries)")
print(f"Every query has >=1 relevance judgment: "
      f"{queries_df['query_id'].isin(qrels_df['query_id']).all()}")

assert meets_corpus_min, "Corpus below the 1,000-passage minimum"
assert meets_query_min, "Query set below the 50-query minimum"
print(f"\n[OK] SciFact data (source={data_source!r}) clears both minimums.")

Corpus  >= 1000: True  (5183 passages)
Queries >= 50: True  (300 queries)
Every query has >=1 relevance judgment: True

[OK] SciFact data (source='local-cache') clears both minimums.


In [4]:
# Inspect a sample of each piece
print("=== Sample corpus passage ===")
print(corpus_df.iloc[0].to_dict())

print("\n=== Sample query ===")
print(queries_df.iloc[0].to_dict())

print("\n=== Sample relevance judgments (qrels) ===")
print(qrels_df.head())

=== Sample corpus passage ===
{'doc_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior 

In [5]:
# Persist to disk so Tasks 2+ (embedding generation, retrieval, ANN search)
# can load the same corpus/queries/qrels without re-running this cell.
import os
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

corpus_df.to_json(f"{DATA_DIR}/corpus.jsonl", orient="records", lines=True)
queries_df.to_json(f"{DATA_DIR}/queries.jsonl", orient="records", lines=True)
qrels_df.to_csv(f"{DATA_DIR}/qrels.tsv", sep="\t", index=False)

print(f"Saved to {DATA_DIR}/: corpus.jsonl, queries.jsonl, qrels.tsv (source={data_source})")

Saved to data/: corpus.jsonl, queries.jsonl, qrels.tsv (source=local-cache)


### Dataset Details and Source

- **Name:** SciFact (from the BEIR benchmark)
- **Domain:** scientific-claim verification abstracts (biomedical / life sciences)
- **Source:** Wadden et al., *"Fact or Fiction: Verifying Scientific Claims"*, EMNLP 2020; redistributed in BEIR format by Thakur et al., *"BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models"*, NeurIPS 2021 (Datasets & Benchmarks track)
- **Access:** `BeIR/scifact` (corpus, queries) and `BeIR/scifact-qrels` (relevance judgments) on the Hugging Face Hub
- **Size (full dataset):** ~5,183 corpus passages, ~300 queries (test split), with binary relevance judgments (qrels) linking each query to its relevant document(s)
- **Format:** corpus = `{doc_id, title, text}`; queries = `{query_id, text}`; qrels = `{query_id, doc_id, relevance}`

### Explanation of the Logic Used

`load_beir_scifact()` checks a local cache under `data/` first, falling back to a live Hugging Face download only if the cache is missing -- both normalize to a consistent `doc_id` / `query_id` schema. The query set is then filtered down to only the query_ids that actually appear in `qrels`: the raw BEIR `queries` split ships **1,109** queries (spanning train/dev/test), but only **300** have a relevance judgment attached, so keeping the unfiltered set would have violated "relevance information for each query."

### Justification for the Chosen Approach

BEIR-format datasets bundle corpus, queries, *and* relevance labels in one consistent schema, which the assignment explicitly permits ("students may use an existing dataset containing query-document relevance labels") and which removes an entire class of errors (inconsistent manual relevance judgments) that a self-built dataset would risk. SciFact specifically was chosen over larger BEIR datasets (e.g. FiQA-2018) because its domain (scientific claims) gives clean, unambiguous relevance judgments — useful later in Task 3 and Task 7 when similarity rankings and qualitative differences need to be explained against ground truth that isn't itself noisy.

### Inference

The loaded corpus (5,183 passages) and filtered query set (300 queries, every one with >=1 relevance judgment) both exceed the assignment's minimums by a wide margin -- **~5x** the corpus minimum, **~6x** the query minimum -- which leaves comfortable room to subsample later if compute time becomes a constraint in Tasks 2/5 without ever dropping below 1,000 passages or 50 queries. The 809 queries dropped for having no qrel are a real, expected feature of BEIR datasets (unlabelled queries exist for other splits/purposes), not a data quality problem.

### Limitations Observed

- SciFact's relevance judgments are binary (relevant / not relevant) with no graded relevance — fine for Recall@5 in Tasks 4–6, but it means Task 3's "which metric appears most suitable" analysis has less ranking nuance to work with than a graded-relevance dataset would give.
- Only 283 of the 5,183 corpus documents are ever marked relevant to any query (per the qrels), which is expected for a claim-verification benchmark (most abstracts are irrelevant to any single claim) but worth remembering when interpreting Recall@5 in later tasks — the "needle in haystack" ratio is real, not an artifact of subsampling.

### Possible Improvements

- Cross-check a second BEIR dataset (e.g. NFCorpus) as a robustness check on Task 7's qualitative findings, time permitting.
- If Task 5/6 index-building is slow at full scale, subsample the corpus in a stratified way (keeping every relevant document for every query) rather than a pure random subsample.

### References

- Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS Datasets & Benchmarks.
- Wadden, D. et al. (2020). *Fact or Fiction: Verifying Scientific Claims.* EMNLP.
- Dataset card: `https://huggingface.co/datasets/BeIR/scifact`

---
# Task 2 — Embedding Generation and Pooling (0.5 Marks)

**Two encoder-based models chosen, deliberately with different profiles:**

1. **`distilbert-base-uncased`** (Sanh et al., 2019) — a general-purpose distilled BERT encoder, **mean pooling**, not specifically trained for retrieval/similarity.
2. **`BAAI/bge-large-en-v1.5`** (Xiao et al., 2023) — an encoder trained specifically for retrieval via RetroMAE pretraining + contrastive fine-tuning, **[CLS]-token pooling**.

## Why encoder models are appropriate for semantic representation and retrieval

An encoder-only transformer (the BERT family) processes the whole input in one forward pass using **bidirectional self-attention** — every token's representation is built from *both* the tokens before and after it. Pooling those token representations (mean pooling, or the `[CLS]` token) yields a single fixed-length dense vector that summarizes the entire input's meaning. That is exactly what semantic retrieval needs: a way to place any query or document into one shared vector space where "similar meaning" becomes "small distance" (cosine / dot-product / L2).

A **decoder-only** (autoregressive) model, by contrast, is trained to predict the *next* token from only the *preceding* context (causal, one-directional attention). It is excellent at generation, but no single hidden state naturally summarizes the whole sequence — extracting a robust sentence embedding from a decoder needs extra engineering (e.g. using only the last token's hidden state, which never attended to anything after it). An encoder-decoder model can also produce embeddings, but carries the overhead of a decoder stage that retrieval doesn't need. Encoder-only models are therefore the efficient, standard choice for embedding generation.

In [6]:
import time
import numpy as np
import pandas as pd

# Documented facts about each model (from official model cards / papers) --
# these don't require running the model, only the embeddings themselves and
# the timing do.
MODEL_SPECS = {
    "distilbert-base-uncased": {"pooling": "mean pooling", "dim": 768, "max_input_length": 512},
    "BAAI/bge-large-en-v1.5": {"pooling": "[CLS] token pooling", "dim": 1024, "max_input_length": 512},
}

corpus_texts = corpus_df["text"].tolist()
comparison_rows = []

for model_name, spec in MODEL_SPECS.items():
    print(f"=== {model_name} ===")
    t0 = time.time()
    elapsed = None
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name)
        embeddings = model.encode(corpus_texts, show_progress_bar=False)
        elapsed = time.time() - t0
        np.save(f"data/embeddings_{model_name.replace('/', '__')}.npy", embeddings)
        print(f"  Generated real embeddings: shape={embeddings.shape}, time={elapsed:.2f}s")
    except Exception as e:
        print(f"  [warning] Could not run this model in the current environment: {e!r}")
        print(f"  Re-run this cell on the remote system (needs sentence-transformers + "
              f"internet access) for real embeddings and timing.")

    comparison_rows.append({
        "Model name": model_name,
        "Embedding dimension": spec["dim"],
        "Max/typical input length (tokens)": spec["max_input_length"],
        "Pooling strategy": spec["pooling"],
        "Approx. time to embed corpus (s)": round(elapsed, 2) if elapsed is not None else "N/A (not run here)",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("data/task2_model_comparison.csv", index=False)
comparison_df

=== distilbert-base-uncased ===
  [warning] Could not run this model in the current environment: ModuleNotFoundError("No module named 'sentence_transformers'")
  Re-run this cell on the remote system (needs sentence-transformers + internet access) for real embeddings and timing.
=== BAAI/bge-large-en-v1.5 ===
  [warning] Could not run this model in the current environment: ModuleNotFoundError("No module named 'sentence_transformers'")
  Re-run this cell on the remote system (needs sentence-transformers + internet access) for real embeddings and timing.


,Model name,Embedding dimension,Max/typical input length (tokens),Pooling strategy,Approx. time to embed corpus (s)
0,distilbert-base-uncased,768,512,mean pooling,N/A (not run here)
1,BAAI/bge-large-en-v1.5,1024,512,[CLS] token pooling,N/A (not run here)


### Explanation of the Logic Used

Model name, embedding dimension, max input length, and pooling strategy are documented facts from each model's official card/paper (`MODEL_SPECS`) — they don't require successfully running the model. Only the actual embeddings and timing need real execution, so the loop attempts that separately per model and reports `"N/A (not run here)"` for timing if it fails, rather than fabricating a number.

### Justification for the Chosen Models

DistilBERT and BGE-large were chosen specifically because they differ on the axis this problem statement asks about — **semantic quality vs search efficiency** — in two ways: (1) DistilBERT is a smaller, faster, general-purpose encoder never fine-tuned for retrieval, while BGE-large is larger and slower but purpose-built for retrieval via contrastive training on similarity pairs; (2) they use different pooling strategies (mean vs `[CLS]`), which is itself one of the "for each model, document..." requirements. This gives Task 7's qualitative comparison a real, explainable axis of disagreement to analyse later, rather than two near-identical models.

### Inference

On the real models (once run with internet access), BGE-large-en-v1.5's embedding dimension (1024) is larger than DistilBERT's (768), and its contrastive/RetroMAE training objective is expected to produce embeddings that separate semantically related and unrelated passages more cleanly — the concrete evidence for this is Task 7's side-by-side retrieval comparison, not asserted here in the abstract.

### Limitations Observed

- `sentence-transformers` (via its `torch` dependency) could not finish installing within this authoring sandbox's time budget, and this sandbox's network is restricted to `pypi.org` regardless, so the timing column shows `"N/A (not run here)"` rather than a real number. **Re-run this cell on the remote system (which has `torch`/`sentence-transformers` and full internet access) to get real timing figures and embeddings.**

### Possible Improvements

- Batch the encoding call (`model.encode(texts, batch_size=...)`) and report GPU vs CPU timing separately once run on the real remote system, since "approximate time required" is heavily hardware-dependent.
- Add a third, mid-sized model (e.g. `bge-base-en-v1.5`) if time allows, to see whether the quality/speed trade-off is smooth or has a knee.

### References

- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter.* arXiv:1910.01108.
- Xiao, S., Liu, Z., Zhang, P., Muennighoff, N. (2023). *C-Pack: Packaged Resources To Advance General Chinese Embedding.* arXiv:2309.07597. (Official citation for the BGE model family, including the English `bge-large-en-v1.5` variant.)

---
# Task 3 — Similarity Metric Comparison (1.5 Marks)
*Owner: P2 — see `ass-1/ROADMAP.md`, Days 3–4. Not started.*

---
# Task 4 — Exact kNN Baseline (1.5 Marks)
*Owner: P2 — see `ass-1/ROADMAP.md`, Days 3–4. Not started.*

---
# Task 5 — HNSW vs IVF (2 Marks)
*Owner: P3 — see `ass-1/ROADMAP.md`, Days 3–6. Not started.*

---
# Task 6 — ANN Trade-off Analysis (1 Mark)
*Owner: P4 — see `ass-1/ROADMAP.md`, Day 7. Not started.*

---
# Task 7 — Qualitative Retrieval Analysis (2 Marks)
*Owner: P4 — see `ass-1/ROADMAP.md`, Days 3–4. Not started.*

---
# Task 8 — Final Recommendation (1 Mark)
*Owners: P1 + P4 — see `ass-1/ROADMAP.md`, Day 8. Not started.*

---
# Final Conclusion
*To be written once Tasks 2–8 are complete — must cover key observations, strengths, limitations, and possible future improvements (see `ass-1/ROADMAP.md` grading risk flags, #16).*

# References
*Consolidated reference list — add each task's citations here as they're completed.*

- Thakur, N. et al. (2021). BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models. NeurIPS.
- Wadden, D. et al. (2020). Fact or Fiction: Verifying Scientific Claims. EMNLP.